In [ ]:
#일반 Bert 텍스트 임베딩
import pandas as pd
import numpy as np
import torch
from transformers import BertTokenizer, BertModel
from tqdm import tqdm  #진행 상황 시각화용

#모델 및 토크나이저 로드
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)

#추론 모드 전환
model.eval() 

def embed_reviews_pure_bert(file_path, column_name):
    #CSV 파일 로드
    df = pd.read_parquet(file_path)
    print(f"파일 로드 완료: {len(df)}개의 행을 발견했습니다.")

    #혹시 모를 결측치 처리
    reviews_df = df[column_name].fillna("")
    reviews = reviews_df.tolist()

    all_embeddings = []

    print("BERT 임베딩 생성 중..")
    
    with torch.no_grad():  #기울기 계산 비활성화 (메모리 절약)
        for review in tqdm(reviews):
            #각각의 문장에 토크나이저 적용
            inputs = tokenizer(review, return_tensors="pt")
            
            #BERT 모델 통과
            outputs = model(**inputs)
            
            #여기서 문장 전체의 의미를 담고 있는 맨 첫 번째 [CLS] 토큰의 벡터만 쏙 뽑아낸다.
            cls_embedding = outputs.last_hidden_state[0][0].numpy() 
            
            all_embeddings.append(cls_embedding)

    
    embeddings_array = np.array(all_embeddings)
    print(f"임베딩 완료! 결과 형태: {embeddings_array.shape}")
    
    return df, embeddings_array


file_path = '../data/data_yelp.parquet'
column_name = 'text'

df, review_embeddings = embed_reviews_pure_bert(file_path, column_name)


df['review_embedding'] = review_embeddings.tolist()
df.to_parquet('pure_Bert_embedding4.parquet', index=False)
df.to_csv('reviews_with_pure_Bert_embeddings.csv', index=False)
print("완료!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


파일 로드 완료: 5000개의 행을 발견했습니다.
순정 BERT 임베딩 생성 중..


100%|██████████| 5000/5000 [14:37<00:00,  5.70it/s]


임베딩 완료! 결과 형태: (5000, 768)
완료!


In [ ]:
#수업 버트모델 임베딩 결과 하이퍼파라미터 검사
import pandas as pd
import numpy as np
import ast
import os
import random
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
import itertools

#글로벌 시드 고정 함수
#학습 도중 파이썬 기본 기능을 이용해 리스트를 다루거나, 샘플을 임의로 뽑아내는 연산이 일어날 때 42번 시드 규칙대로 움직이도록 고정
def set_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def set_randomseeds():
    #고정되어 있던 글로벌 시드 락을 완전히 해제하여 다음 실험을 순수 랜덤 상태로 리셋
    random.seed(None)
    np.random.seed(None)
    tf.random.set_seed(None)

#데이터 로드 및 전처리
def prepare_data(file_path):
    print(f"'{file_path}' 데이터 로딩 중...")
    df = pd.read_parquet(file_path)
    X = np.array([ast.literal_eval(x) for x in df['review_embedding']])
    #y = df['label'].values
    y = df['label'].map({'ai': 1, 'human': 0}).values


    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y, test_size=0.15, random_state=42
    )
    
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=(0.15 / 0.85), random_state=42
    )
    
    return X_train, X_val, X_test, y_train, y_val, y_test

# 하이퍼파라미터 조합
batch_sizes = [32, 64, 128]
dropout_rates = [0.0, 0.1, 0.3] 
d_models = [128, 256, 512]
learning_rates = [0.00003, 0.0001, 0.0003]
configs = list(itertools.product(batch_sizes, dropout_rates, d_models, learning_rates))

#데이터 준비
X_train, X_val, X_test, y_train, y_val, y_test = prepare_data('reviews_with_pure_Bert_embeddings.csv')
final_results = []

X_train_val = np.concatenate([X_train, X_val], axis=0)
y_train_val = np.concatenate([y_train, y_val], axis=0)

#중단 설정
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0)

print(f"총 {len(configs)}개 조합 딥러닝 실험 시작")

for idx, (b_size, drop_rate, d_model, lr) in enumerate(configs):
    
    print(f"[{idx+1}/{len(configs)}] 실험 중: Batch:{b_size}, Dropout:{drop_rate}, D_Model:{d_model}, LR:{lr}")
    
    #고정 시드(42) 실험
    set_seeds(42) #난수 엔진 42로 고정 Lock(fit을 했을때 42시드의 양식을 따라가도록)
    initializer_fix = tf.keras.initializers.GlorotUniform(seed=42)
    
    model_fixed = models.Sequential([
        layers.Input(shape=(768,)),
        layers.Dense(d_model, activation='relu', kernel_initializer=initializer_fix),
        layers.Dropout(drop_rate, seed=42),
        layers.Dense(1, activation='sigmoid', kernel_initializer=initializer_fix)
    ])
    
    model_fixed.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    model_fixed.fit(
        X_train_val, y_train_val, 
        validation_data=(X_val, y_val), 
        batch_size=b_size, epochs=30, callbacks=[early_stopping], verbose=0
    )
     #손실값과 정확도값 추출 이때 손실값은 _에 의해 추출하지 않는다.
    _, fixed_acc = model_fixed.evaluate(X_test, y_test, verbose=0)
    
    #랜덤 시드 실험
    #메모리 초기화 + 시드 락 풀기(기존의 42시드로 고정되어 있던 fit 환경을 초기화 한다.) 
    tf.keras.backend.clear_session()
    set_randomseeds()
    #랜덤 시드 하나 생성
    initializer_rand = tf.keras.initializers.GlorotUniform(seed=None)
    
    model_rand = models.Sequential([
        layers.Input(shape=(768,)),
        layers.Dense(d_model, activation='relu', kernel_initializer=initializer_rand),
        layers.Dropout(drop_rate, seed=None),
        layers.Dense(1, activation='sigmoid', kernel_initializer=initializer_rand)
    ])
    
    model_rand.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    model_rand.fit(
        X_train_val, y_train_val, 
        validation_data=(X_val, y_val), 
        batch_size=b_size, epochs=30, callbacks=[early_stopping], verbose=0
    )
    #손실값과 정확도값 추출 이때 손실값은 _에 의해 추출하지 않는다.
    _, random_acc = model_rand.evaluate(X_test, y_test, verbose=0)

    #데이터 정리
    res = {
        'd_model': d_model, 'batch_size': b_size, 'dropout': drop_rate, 'lr': lr,
        'fixed_accuracy': fixed_acc,    # 고정 시드 점수
        'random_accuracy': random_acc   # 랜덤 시드 점수
    }
    final_results.append(res)

#CSV 저장
results_df = pd.DataFrame(final_results)
results_df.to_csv('Pure_bert_experiment_report.csv', index=False)
print("\n[완료] 'Pure_bert_experiment_report.csv' 파일이 성공적으로 생성되었습니다.")

I0000 00:00:1779978173.864307    1077 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779978191.321561    1077 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


'reviews_with_pure_Bert_embeddings.csv' 데이터 로딩 중...
총 81개 조합 딥러닝 실험 시작
[1/81] 실험 중... Batch:32, Dropout:0.0, D_Model:128, LR:3e-05


E0000 00:00:1779978266.273090    1077 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
W0000 00:00:1779978267.983195    1077 cpu_allocator_impl.cc:82] Allocation of 52224000 exceeds 10% of free system memory.
W0000 00:00:1779978343.263006    1077 cpu_allocator_impl.cc:82] Allocation of 52224000 exceeds 10% of free system memory.


[2/81] 실험 중... Batch:32, Dropout:0.0, D_Model:128, LR:0.0001


W0000 00:00:1779978357.168177    1077 cpu_allocator_impl.cc:82] Allocation of 52224000 exceeds 10% of free system memory.
W0000 00:00:1779978371.933207    1077 cpu_allocator_impl.cc:82] Allocation of 52224000 exceeds 10% of free system memory.


[3/81] 실험 중... Batch:32, Dropout:0.0, D_Model:128, LR:0.0003


W0000 00:00:1779978385.553529    1077 cpu_allocator_impl.cc:82] Allocation of 52224000 exceeds 10% of free system memory.


[4/81] 실험 중... Batch:32, Dropout:0.0, D_Model:256, LR:3e-05
[5/81] 실험 중... Batch:32, Dropout:0.0, D_Model:256, LR:0.0001
[6/81] 실험 중... Batch:32, Dropout:0.0, D_Model:256, LR:0.0003
[7/81] 실험 중... Batch:32, Dropout:0.0, D_Model:512, LR:3e-05
[8/81] 실험 중... Batch:32, Dropout:0.0, D_Model:512, LR:0.0001
[9/81] 실험 중... Batch:32, Dropout:0.0, D_Model:512, LR:0.0003
[10/81] 실험 중... Batch:32, Dropout:0.1, D_Model:128, LR:3e-05
[11/81] 실험 중... Batch:32, Dropout:0.1, D_Model:128, LR:0.0001
[12/81] 실험 중... Batch:32, Dropout:0.1, D_Model:128, LR:0.0003
[13/81] 실험 중... Batch:32, Dropout:0.1, D_Model:256, LR:3e-05
[14/81] 실험 중... Batch:32, Dropout:0.1, D_Model:256, LR:0.0001
[15/81] 실험 중... Batch:32, Dropout:0.1, D_Model:256, LR:0.0003
[16/81] 실험 중... Batch:32, Dropout:0.1, D_Model:512, LR:3e-05
[17/81] 실험 중... Batch:32, Dropout:0.1, D_Model:512, LR:0.0001
[18/81] 실험 중... Batch:32, Dropout:0.1, D_Model:512, LR:0.0003
[19/81] 실험 중... Batch:32, Dropout:0.3, D_Model:128, LR:3e-05
[20/81] 실험 중... Batc

In [ ]:
import pandas as pd
import numpy as np

report_file = 'Pure_bert_experiment_report.csv'
print(f"'{report_file}' 결과 보고서 분석 중...\n")

#실험 결과 파일 로드
df = pd.read_csv(report_file)

#안정성 평가를 위한 고정-랜덤 격차 계산
df['accuracy_diff'] = (df['fixed_accuracy'] - df['random_accuracy']).abs()

#종합 점수 계산 (랜덤 점수 기반 + 격차 페널티(0.5) 반영)
df['total_score'] = df['random_accuracy'] - (df['accuracy_diff'] * 0.5)

#종합 점수 높은 순, 랜덤 정확도 높은 순, 모델 크기(d_model)는 가벼운 순
sorted_df = df.sort_values(
    by=['total_score', 'random_accuracy', 'd_model'], 
    ascending=[False, False, True]
)

#최적 조합 하나 추출
best_config = sorted_df.iloc[0]

#화면 출력
print("최종 조합")
print(f"Batch Size: {int(best_config['batch_size'])}")
print(f"Dropout Rate: {best_config['dropout']}")
print(f"D_Model: {int(best_config['d_model'])}")
print(f"Learning-Rate: {best_config['lr']}")
print("-----------------------------------------------------------------------")
print("성능 지표")
print(f"랜덤 시드 정확도: {best_config['random_accuracy']:.4f}")
print(f"고정 시드 정확도: {best_config['fixed_accuracy']:.4f}")
print(f"두 시드 간 점수 격차: {best_config['accuracy_diff']:.4f}")

#전체 순위 파일도 csv 파일로 저장
sorted_df.to_csv('Pure_bert_analyzed_ranking.csv', index=False)

'Pure_bert_experiment_report.csv' 결과 보고서 분석 중...

 최종 1위 조합
Batch Size: 32
Dropout Rate: 0.1
D_Model: 512
Learning-Rate: 0.0003
-----------------------------------------------------------------------
성능 지표
랜덤 시드 정확도: 0.9707
고정 시드 정확도: 0.9700
두 시드 간 점수 격차: 0.0007


In [ ]:
#최적 하이퍼파라미터 조합 각 시드별 실험
import numpy as np
import pandas as pd
import ast
import random
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

#글로벌 시드 고정 함수 (42,43,44,45,46)
def set_seeds(seed=46): #
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

#파일 실행하자마자 시스템 전체 난수 자물쇠 잠그기
set_seeds(46) #

#데이터 로드 및 분할
df = pd.read_csv('reviews_with_pure_Bert_embeddings.csv')
X = np.array([ast.literal_eval(x) for x in df['review_embedding']])
y = df['label'].map({'ai': 1, 'human': 0}).values 

#데이터 분할
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=46 #
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=(0.15 / 0.85), random_state=46 #
)

#최적의 하이퍼파라미터 규격 설정
best_d_model = 512     
best_batch_size = 32     
best_dropout = 0.1
best_lr = 0.0003         

#시드가 적용된 최종 Deep MLP 모델 빌드
initializer_fix = tf.keras.initializers.GlorotUniform(seed=46) #

model = models.Sequential([
    layers.Input(shape=(768,)),
    layers.Dense(best_d_model, activation='relu', kernel_initializer=initializer_fix),
    layers.Dropout(best_dropout, seed=46), # 
    layers.Dense(1, activation='sigmoid', kernel_initializer=initializer_fix)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=best_lr),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

#조기 종료 설정
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True, verbose=0
)

#최종 모델 학습 시작
print("최적의 하이퍼파라미터 학습 중")
model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=best_batch_size,
    epochs=30,
    callbacks=[early_stopping],
    verbose=0
)

#예측 및 확률값 추출
y_prob = model.predict(X_test, verbose=0).flatten()
y_pred = (y_prob >= 0.5).astype(int)

#성능 평가지표 계산
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auroc = roc_auc_score(y_test, y_prob)

print("최종 모델 평가 결과")
print(f"Accuracy:  {accuracy:.4f}  ")
print(f"Precision: {precision:.4f} ") 
print(f"Recall:    {recall:.4f} ")
print(f"F1 Score:  {f1:.4f} ")
print(f"AUROC:     {auroc:.4f} ")
print(f"=========================================")

#print("\n클래스별 상세 보고서:")
#print(classification_report(y_test, y_pred, target_names=['human', 'ai']))

최적의 하이퍼파라미터 학습 중


W0000 00:00:1780039183.110240    1286 cpu_allocator_impl.cc:82] Allocation of 43008000 exceeds 10% of free system memory.


최종 모델 평가 결과
Accuracy:  0.9767  
Precision: 0.9753 
Recall:    0.9766 
F1 Score:  0.9760 
AUROC:     0.9975 


In [ ]:
#최신 모델 Bert 임베딩
import pandas as pd
#허깅 페이스에서 임베딩 모델과 일반 모델 호출을 도와주는 패키지 임포트
#transformers 모델의 확장판으로 일반 모델 호출도 가능합니다. (임베딩 추출에 특화되어 있어 사용)
from sentence_transformers import SentenceTransformer, models
#탠서 처리를 위한 패키지
import torch
import os

#모델 설정 (2026년 최신 ModernBERT)
#answerdotai/ModernBERT-base는 기본적으로 Masked LM 모델이므로, 
#Sentence-Transformers에서(임베딩 벡터를 뽑아내기 위해) 사용하기 위해 Mean Pooling 레이어를 수동으로 구성해줘야 했습니다.
model_id = 'answerdotai/ModernBERT-base'

#출력층을 제거한 ModernBERT 모델 호출
word_embedding_model = models.Transformer(model_id, max_seq_length=512)
#임베딩 벡터를 추출하기 위한 풀링층
pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), pooling_mode='mean')
#입력에 따른 임베딩 백터를 추출하기 위한 모델 구성
model = SentenceTransformer(modules=[word_embedding_model, pooling_model])

def embed_reviews(file_path, column_name):
    # Parquet 파일 불러오기
    df = pd.read_parquet(file_path)
    print(f"파일 로드 완료: {len(df)}개의 행을 발견했습니다.")

    # 결측치 처리
    reviews_df = df[column_name].fillna("")    
    reviews = reviews_df.tolist()

    #임베딩 생성
    print("\nModernBERT 임베딩 생성 중:")
    embeddings = model.encode(
        reviews, 
        batch_size=32,
        show_progress_bar=True, 
        convert_to_numpy=True, 
        device='cpu'            #GPU 미사용 명시 (현재 제 환경은 GPU 사용이 불가합니다...)
    )

    print(f"임베딩 완료! 결과 형태: {embeddings.shape}") #(768 차원)
    
    return df, embeddings


file_path = 'reviews_part4.csv'
column_name = 'text'

df, review_embeddings_modern = embed_reviews(file_path, column_name)

if df is not None:
    #임베딩 결과를 리스트로 변환하여 저장
    df['review_embedding'] = review_embeddings_modern.tolist()
    
    #결과 저장
    output_name = 'ModernBERT_embeddings4.csv'
    df.to_csv(output_name, index=False)
    print(f"최종 결과가 '{output_name}'으로 저장되었습니다.")

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_88721/4214715816.py:18: FutureWarning: The `get_word_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), pooling_mode='mean')


파일 로드 완료: 5000개의 행을 발견했습니다.

ModernBERT 임베딩 생성 중... (CPU 환경이므로 시간이 소요될 수 있습니다)


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

임베딩 완료! 결과 형태: (5000, 768)
최종 결과가 'ModernBERT_embeddings4.csv'으로 저장되었습니다.


In [ ]:
#최신 버트모델 임베딩 결과 하이퍼파라미터 검사
import pandas as pd
import numpy as np
import ast
import os
import random
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
import itertools

#글로벌 시드 고정 함수
#학습 도중 파이썬 기본 기능을 이용해 리스트를 다루거나, 샘플을 임의로 뽑아내는 연산이 일어날 때 42번 시드 규칙대로만 움직이도록 고정
def set_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def set_randomseeds():
    #고정되어 있던 글로벌 시드 락(Lock)을 완전히 해제하여 다음 실험을 순수 랜덤 상태로 리셋
    random.seed(None)
    np.random.seed(None)
    tf.random.set_seed(None)

#데이터 로드 및 전처리
def prepare_data(file_path):
    print(f"'{file_path}' 데이터 로딩 중...")
    df = pd.read_parquet(file_path)
    X = np.array([ast.literal_eval(x) for x in df['review_embedding']])
    #y = df['label'].values
    y = df['label'].map({'ai': 1, 'human': 0}).values


    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y, test_size=0.15, random_state=42
    )
    
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=(0.15 / 0.85), random_state=42
    )
    
    return X_train, X_val, X_test, y_train, y_val, y_test

#하이퍼파라미터 조합
batch_sizes = [32, 64, 128]
dropout_rates = [0.0, 0.1, 0.3] 
d_models = [128, 256, 512]
learning_rates = [0.00003, 0.0001, 0.0003]
configs = list(itertools.product(batch_sizes, dropout_rates, d_models, learning_rates))

#데이터 준비
X_train, X_val, X_test, y_train, y_val, y_test = prepare_data('reviews_with_Modern_Bert_embeddings.csv')
final_results = []

X_train_val = np.concatenate([X_train, X_val], axis=0)
y_train_val = np.concatenate([y_train, y_val], axis=0)

#중단 설정
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True, verbose=0
)

print(f"총 {len(configs)}개 조합 딥러닝 실험 시작")

for idx, (b_size, drop_rate, d_model, lr) in enumerate(configs):
    
    print(f"[{idx+1}/{len(configs)}] 실험 중: Batch:{b_size}, Dropout:{drop_rate}, D_Model:{d_model}, LR:{lr}")
    
    #고정 시드 실험
    set_seeds(42)  #난수 엔진 42로 고정 Lock (fit을 했을때 42시드의 양식을 따라가도록)
    initializer_fix = tf.keras.initializers.GlorotUniform(seed=42)
    
    model_fixed = models.Sequential([
        layers.Input(shape=(768,)),
        layers.Dense(d_model, activation='relu', kernel_initializer=initializer_fix),
        layers.Dropout(drop_rate, seed=42),
        layers.Dense(1, activation='sigmoid', kernel_initializer=initializer_fix)
    ])
    
    model_fixed.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    model_fixed.fit(
        X_train_val, y_train_val, 
        validation_data=(X_val, y_val), 
        batch_size=b_size, epochs=30, callbacks=[early_stopping], verbose=0
    )
     #손실값과 정확도값 추출 이때 손실값은 _에 의해 추출하지 않는다.
    _, fixed_acc = model_fixed.evaluate(X_test, y_test, verbose=0)
    
    #랜덤 시드 실험
    #메모리 누수 방지 + 시드 락 풀기(기존의 42시드로 고정되어 있던 fit 환경을 초기화 한다.) 
    tf.keras.backend.clear_session()
    set_randomseeds()
    #랜덤 시드 하나 생성
    initializer_rand = tf.keras.initializers.GlorotUniform(seed=None)
    
    model_rand = models.Sequential([
        layers.Input(shape=(768,)),
        layers.Dense(d_model, activation='relu', kernel_initializer=initializer_rand),
        layers.Dropout(drop_rate, seed=None),
        layers.Dense(1, activation='sigmoid', kernel_initializer=initializer_rand)
    ])
    
    model_rand.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    model_rand.fit(
        X_train_val, y_train_val, 
        validation_data=(X_val, y_val), 
        batch_size=b_size, epochs=30, callbacks=[early_stopping], verbose=0
    )
    #손실값과 정확도값 추출 이때 손실값은 _에 의해 추출하지 않는다.
    _, random_acc = model_rand.evaluate(X_test, y_test, verbose=0)

    #데이터 정리
    res = {
        'd_model': d_model, 'batch_size': b_size, 'dropout': drop_rate, 'lr': lr,
        'fixed_accuracy': fixed_acc,    # 고정 시드 점수
        'random_accuracy': random_acc   # 랜덤 시드 점수
    }
    final_results.append(res)

#CSV 저장
results_df = pd.DataFrame(final_results)
results_df.to_csv('Modern_bert_experiment_report.csv', index=False)
print("\n[완료] 'Modern_bert_experiment_report.csv' 파일이 성공적으로 생성되었습니다.")

I0000 00:00:1779987005.985378   88721 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779987021.891563   88721 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


'reviews_with_Modern_Bert_embeddings.csv' 데이터 로딩 중...
총 81개 조합 딥러닝 실험 시작
[1/81] 실험 중... Batch:32, Dropout:0.0, D_Model:128, LR:3e-05


E0000 00:00:1779987103.474129   88721 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


[2/81] 실험 중... Batch:32, Dropout:0.0, D_Model:128, LR:0.0001
[3/81] 실험 중... Batch:32, Dropout:0.0, D_Model:128, LR:0.0003
[4/81] 실험 중... Batch:32, Dropout:0.0, D_Model:256, LR:3e-05
[5/81] 실험 중... Batch:32, Dropout:0.0, D_Model:256, LR:0.0001
[6/81] 실험 중... Batch:32, Dropout:0.0, D_Model:256, LR:0.0003
[7/81] 실험 중... Batch:32, Dropout:0.0, D_Model:512, LR:3e-05
[8/81] 실험 중... Batch:32, Dropout:0.0, D_Model:512, LR:0.0001
[9/81] 실험 중... Batch:32, Dropout:0.0, D_Model:512, LR:0.0003
[10/81] 실험 중... Batch:32, Dropout:0.1, D_Model:128, LR:3e-05
[11/81] 실험 중... Batch:32, Dropout:0.1, D_Model:128, LR:0.0001
[12/81] 실험 중... Batch:32, Dropout:0.1, D_Model:128, LR:0.0003
[13/81] 실험 중... Batch:32, Dropout:0.1, D_Model:256, LR:3e-05
[14/81] 실험 중... Batch:32, Dropout:0.1, D_Model:256, LR:0.0001
[15/81] 실험 중... Batch:32, Dropout:0.1, D_Model:256, LR:0.0003
[16/81] 실험 중... Batch:32, Dropout:0.1, D_Model:512, LR:3e-05
[17/81] 실험 중... Batch:32, Dropout:0.1, D_Model:512, LR:0.0001
[18/81] 실험 중... Batch

In [ ]:
import pandas as pd
import numpy as np

report_file = 'Modern_bert_experiment_report.csv'
print(f"'{report_file}' 결과 보고서 분석 중...\n")

#실험 결과 파일 로드
df = pd.read_csv(report_file)

#안정성 평가를 위한 고정-랜덤 격차 계산
df['accuracy_diff'] = (df['fixed_accuracy'] - df['random_accuracy']).abs()

#종합 점수 계산 (랜덤 점수 기반 + 격차 페널티 반영)
df['total_score'] = df['random_accuracy'] - (df['accuracy_diff'] * 0.5)

#종합 점수 높은 순, 랜덤 정확도 높은 순, 모델 크기(d_model)는 가벼운 순
sorted_df = df.sort_values(
    by=['total_score', 'random_accuracy', 'd_model'], 
    ascending=[False, False, True]
)

#최적 조합 하나 추출
best_config = sorted_df.iloc[0]

#화면 출력
print("최종 조합")
print(f"Batch Size: {int(best_config['batch_size'])}")
print(f"Dropout Rate: {best_config['dropout']}")
print(f"D_Model: {int(best_config['d_model'])}")
print(f"Learning-Rate: {best_config['lr']}")
print("-----------------------------------------------------------------------")
print("성능 지표")
print(f"랜덤 시드 정확도: {best_config['random_accuracy']:.4f}")
print(f"고정 시드 정확도: {best_config['fixed_accuracy']:.4f}")
print(f"두 시드 간 점수 격차: {best_config['accuracy_diff']:.4f}")

#전체 순위 파일 저장
sorted_df.to_csv('Modern_bert_analyzed_ranking.csv', index=False)

'Modern_bert_experiment_report.csv' 결과 보고서 분석 중...

 최종 1위 조합
Batch Size: 32
Dropout Rate: 0.0
D_Model: 128
Learning-Rate: 0.0003
-----------------------------------------------------------------------
성능 지표
랜덤 시드 정확도: 0.9823
고정 시드 정확도: 0.9710
두 시드 간 점수 격차: 0.0113


In [ ]:
import numpy as np
import pandas as pd
import ast
import random
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

#글로벌 시드 고정 함수
def set_seeds(seed=42): #
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

#파일 실행하자마자 시스템 전체 난수 자물쇠 잠그기
set_seeds(42) #

#데이터 로드 및 분할
df = pd.read_csv('reviews_with_Modern_Bert_embeddings.csv')
X = np.array([ast.literal_eval(x) for x in df['review_embedding']])
y = df['label'].map({'ai': 1, 'human': 0}).values 

#데이터 분할
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42 #
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=(0.15 / 0.85), random_state=42 #
)

#최적의 하이퍼파라미터 규격 설정
best_d_model = 128   
best_batch_size = 32     
best_dropout = 0.0
best_lr = 0.0003         

#시드가 적용된 최종 Deep MLP 모델 빌드
initializer_fix = tf.keras.initializers.GlorotUniform(seed=42) #

model = models.Sequential([
    layers.Input(shape=(768,)),
    layers.Dense(best_d_model, activation='relu', kernel_initializer=initializer_fix),
    layers.Dropout(best_dropout, seed=42), # 
    layers.Dense(1, activation='sigmoid', kernel_initializer=initializer_fix)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=best_lr),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

#조기 종료 설정
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True, verbose=0
)

#최종 모델 학습 시작
print("최적의 하이퍼파라미터 학습 중")
model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=best_batch_size,
    epochs=30,
    callbacks=[early_stopping],
    verbose=0
)

#예측 및 확률값 추출
y_prob = model.predict(X_test, verbose=0).flatten()
y_pred = (y_prob >= 0.5).astype(int)

#성능 평가지표 계산
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auroc = roc_auc_score(y_test, y_prob)

print("최종 모델 평가 결과")
print(f"Accuracy:  {accuracy:.4f}  ")
print(f"Precision: {precision:.4f} ") 
print(f"Recall:    {recall:.4f} ")
print(f"F1 Score:  {f1:.4f} ")
print(f"AUROC:     {auroc:.4f} ")
print(f"=========================================")

#print("\n클래스별 상세 보고서:")
#print(classification_report(y_test, y_pred, target_names=['human', 'ai']))

최적의 하이퍼파라미터 학습 중
최종 모델 평가 결과
Accuracy:  0.9773  
Precision: 0.9671 
Recall:    0.9879 
F1 Score:  0.9774 
AUROC:     0.9981 
